In [ ]:
import os
import json
import torch
from tqdm import tqdm

# Hugging Face & Models
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import PeftModel

# Evaluation Metrics
from sacrebleu.metrics import CHRF, BLEU
import evaluate
from comet import download_model, load_from_checkpoint

# ==========================================
# CẤU HÌNH MÔI TRƯỜNG & CACHE
# ==========================================

BASE_PATH = "/workspace/dongg"

os.environ["HF_HOME"] = f"{BASE_PATH}/hf_cache"
os.environ["HF_DATASETS_CACHE"] = f"{BASE_PATH}/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{BASE_PATH}/hf_cache/models"
os.environ["HF_HUB_CACHE"] = f"{BASE_PATH}/hf_cache/hub"
os.environ["TMPDIR"] = f"{BASE_PATH}/tmp"
torch.set_float32_matmul_precision("high")

for path in ["datasets", "models", "hub"]:
    os.makedirs(f"{BASE_PATH}/hf_cache/{path}", exist_ok=True)
os.makedirs(f"{BASE_PATH}/tmp", exist_ok=True)

# ==========================================
# CÁC HÀM PHỤ TRỢ
# ==========================================
def get_nllb_lang_code(lang):
    lang_map = {
        'fil': 'tgl_Latn', 'khm': 'khm_Khmr', 'lo': 'lao_Laoo',
        'th': 'tha_Thai', 'my': 'mya_Mymr', 'vi': 'vie_Latn',
        'en': 'eng_Latn', 'id': 'ind_Latn', 'ms': 'zsm_Latn',
    }
    return lang_map.get(lang, lang)

def translate_batch(texts, tokenizer, model, src_lang, tgt_lang, device):
    tokenizer.src_lang = get_nllb_lang_code(src_lang)
    tgt_code = get_nllb_lang_code(tgt_lang)

    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code),
            max_new_tokens=256
        )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# ==========================================
# HÀM CHÍNH: DỊCH VÀ ĐÁNH GIÁ ĐỒNG THỜI
# ==========================================
def run_pipeline(dataset, model_name="facebook/nllb-200-3.3B", trans_batch_size=128, eval_batch_size=64, output_file="final_results_test.json"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    other_langs = ['en', 'khm', 'lo', 'th', 'my', 'fil', 'id', 'ms']
    test_data = dataset['test']
    
    # 1. KHỞI TẠO TẤT CẢ MODEL VÀ METRIC TRƯỚC
    print("--- ĐANG KHỞI TẠO MÔ HÌNH DỊCH THUẬT ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    
    PERT = False
    if PERT:
        print("Đang tải PEFT adapter...")
        trans_model = PeftModel.from_pretrained(trans_model, "NllB_3.3B/checkpoint-11000")
    
        trans_model.merge_and_unload()
    trans_model.eval()

    print("\n--- ĐANG KHỞI TẠO CÁC CÔNG CỤ ĐÁNH GIÁ ---")
    chrf = CHRF(word_order=2)
    sbleu = BLEU(effective_order=True) 
    meteor_metric = evaluate.load('meteor')
    
    print("Đang tải mô hình COMET...")
    comet_model_path = download_model("Unbabel/wmt22-comet-da")
    comet_metric = load_from_checkpoint(comet_model_path)
    
    # 2. CẤU TRÚC LƯU TRỮ KẾT QUẢ
    results = {'vi_to_other': {}, 'other_to_vi': {}}
    all_translations_with_scores = []
    
    # 3. CHẠY VÒNG LẶP: DỊCH -> CHẤM ĐIỂM NGAY LẬP TỨC
    for lang in other_langs:
        print(f"\n==============================================")
        print(f"BẮT ĐẦU XỬ LÝ NGÔN NGỮ: {lang.upper()}")
        print(f"==============================================")
        
        # Định nghĩa các chiều dịch
        directions = [
            ('vi_to_other', 'vi', lang),
            ('other_to_vi', lang, 'vi')
        ]
        
        for dir_name, src_lang, tgt_lang in directions:
            print(f"\n--- [1/2] Đang dịch chiều: {src_lang} -> {tgt_lang} ---")
            
            src_texts = test_data[src_lang]
            ref_texts = test_data[tgt_lang]
            pred_texts = []
            
            # Quá trình dịch
            for i in tqdm(range(0, len(src_texts), trans_batch_size), desc=f"Translate {src_lang}->{tgt_lang}"):
                batch_src = src_texts[i:i+trans_batch_size]
                batch_pred = translate_batch(batch_src, tokenizer, trans_model, src_lang, tgt_lang, device)
                pred_texts.extend(batch_pred)
                
            # Giải phóng VRAM nhẹ trước khi chạy đánh giá
            torch.cuda.empty_cache()
            
            print(f"--- [2/2] Đang chấm điểm chiều: {src_lang} -> {tgt_lang} ---")
            
            # Tính độ dài trung bình
            pred_len_words = [len(pred.split()) for pred in pred_texts]
            pred_len_chars = [len(pred) for pred in pred_texts]
            avg_words = sum(pred_len_words) / len(pred_len_words) if pred_len_words else 0
            avg_chars = sum(pred_len_chars) / len(pred_len_chars) if pred_len_chars else 0

            # Chấm điểm truyền thống
            score_sbleu = sbleu.corpus_score(pred_texts, [ref_texts]).score
            score_chrf = chrf.corpus_score(pred_texts, [ref_texts]).score
            score_meteor = meteor_metric.compute(predictions=pred_texts, references=ref_texts)['meteor']
            
            # Chấm điểm COMET
            comet_data = [{"src": src, "mt": mt, "ref": ref} for src, mt, ref in zip(src_texts, pred_texts, ref_texts)]
            comet_output = comet_metric.predict(comet_data, batch_size=eval_batch_size, gpus=1 if torch.cuda.is_available() else 0)
            
            # Lưu kết quả
            results[dir_name][lang] = {
                'SacreBLEU': score_sbleu,
                'chrF++': score_chrf,
                'METEOR': score_meteor,
                'COMET': comet_output.system_score,
                'Avg_Words': avg_words,
                'Avg_Chars': avg_chars
            }
            
            # In kết quả ngay ra màn hình
            print(f"> KẾT QUẢ {src_lang}->{tgt_lang}:")
            print(f"  SacreBLEU : {score_sbleu:.2f}")
            print(f"  chrF++    : {score_chrf:.2f}")
            print(f"  METEOR    : {score_meteor:.4f}")
            print(f"  COMET     : {comet_output.system_score:.4f}")
            print(f"  Avg Len   : {avg_words:.1f} từ / {avg_chars:.1f} ký tự")
            
            # Lưu lịch sử từng câu
            for idx in range(len(src_texts)):
                all_translations_with_scores.append({
                    'src_text': src_texts[idx],
                    'label': ref_texts[idx],
                    'pred_text': pred_texts[idx],
                    'pred_length_words': pred_len_words[idx],
                    'pred_length_chars': pred_len_chars[idx],
                    'other_language': lang,
                    'direction': dir_name,
                    'sentence_comet': comet_output.scores[idx]
                })

            # Backup kết quả liên tục vào file (đề phòng crash giữa chừng)
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump({
                    'scores': results,
                    'translations': all_translations_with_scores
                }, f, ensure_ascii=False, indent=2)

    # 4. TÍNH TRUNG BÌNH TỔNG THỂ KHI XONG TẤT CẢ
    print("\n==============================================")
    print("HOÀN TẤT! TÍNH TOÁN ĐIỂM TRUNG BÌNH TỔNG THỂ")
    
    def calc_avg(direction_dict, metric):
        if len(direction_dict) == 0: return 0
        return sum(d[metric] for d in direction_dict.values()) / len(direction_dict)

    results['average'] = {
        'overall': {
            'SacreBLEU': (calc_avg(results['vi_to_other'], 'SacreBLEU') + calc_avg(results['other_to_vi'], 'SacreBLEU')) / 2,
            'chrF++': (calc_avg(results['vi_to_other'], 'chrF++') + calc_avg(results['other_to_vi'], 'chrF++')) / 2,
            'METEOR': (calc_avg(results['vi_to_other'], 'METEOR') + calc_avg(results['other_to_vi'], 'METEOR')) / 2,
            'COMET': (calc_avg(results['vi_to_other'], 'COMET') + calc_avg(results['other_to_vi'], 'COMET')) / 2,
        }
    }
    
    # Ghi đè file với điểm trung bình cuối cùng
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump({
            'scores': results,
            'translations': all_translations_with_scores
        }, f, ensure_ascii=False, indent=2)
        
    print(f"Đã lưu toàn bộ kết quả vào: {output_file}")
    print(f"Điểm COMET trung bình tổng quát: {results['average']['overall']['COMET']:.4f}")

# ==========================================
# THỰC THI CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    DATASET_FILE = "DATA/ALT/alt_multilingual_dataset/test.jsonl"
    FINAL_OUTPUT_FILE = "TESSSSSSSSSSSSSSSSSSSSSSST/final_results_goc.json"
    
    print("Đang tải dữ liệu từ file JSONL...")
    dataset = load_dataset("json", data_files={"test": DATASET_FILE})
    
    run_pipeline(
        dataset=dataset,
        model_name="facebook/nllb-200-3.3B",
        trans_batch_size=128,  # Batch size cho bước dịch
        eval_batch_size=64,    # Batch size cho COMET
        output_file=FINAL_OUTPUT_FILE
    )

# Pivot

In [ ]:
import os
import json
import torch
from tqdm import tqdm

# Hugging Face & Models
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import PeftModel

# Evaluation Metrics
from sacrebleu.metrics import CHRF, BLEU
from comet import download_model, load_from_checkpoint

import warnings
import logging

# 1. Tắt toàn bộ các cảnh báo (UserWarning) từ Python
warnings.filterwarnings("ignore")

# 2. Tắt log hệ thống của PyTorch Lightning & Fabric (ẩn srun, GPU info, tips...)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)

# 3. Tắt cảnh báo của sacrebleu (ẩn dòng lặp đi lặp lại "effective_order")
logging.getLogger("sacrebleu").setLevel(logging.ERROR)

# Tắt thêm log của mô hình transformers (nếu cần)
logging.getLogger("transformers").setLevel(logging.ERROR)

# ==========================================
# CẤU HÌNH MÔI TRƯỜNG & CACHE
# ==========================================

BASE_PATH = "/workspace/dongg"

os.environ["HF_HOME"] = f"{BASE_PATH}/hf_cache"
os.environ["HF_DATASETS_CACHE"] = f"{BASE_PATH}/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{BASE_PATH}/hf_cache/models"
os.environ["HF_HUB_CACHE"] = f"{BASE_PATH}/hf_cache/hub"
os.environ["TMPDIR"] = f"{BASE_PATH}/tmp"
torch.set_float32_matmul_precision("high")

for path in ["datasets", "models", "hub"]:
    os.makedirs(f"{BASE_PATH}/hf_cache/{path}", exist_ok=True)
os.makedirs(f"{BASE_PATH}/tmp", exist_ok=True)

# ==========================================
# CÁC HÀM PHỤ TRỢ
# ==========================================
def get_nllb_lang_code(lang):
    lang_map = {
        'fil': 'tgl_Latn', 'khm': 'khm_Khmr', 'lo': 'lao_Laoo',
        'th': 'tha_Thai', 'my': 'mya_Mymr', 'vi': 'vie_Latn',
        'en': 'eng_Latn', 'id': 'ind_Latn', 'ms': 'zsm_Latn',
    }
    return lang_map.get(lang, lang)

def translate_batch(texts, tokenizer, model, src_lang, tgt_lang, device):
    tokenizer.src_lang = get_nllb_lang_code(src_lang)
    tgt_code = get_nllb_lang_code(tgt_lang)

    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code),
            max_new_tokens=256
        )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# ==========================================
# HÀM CHÍNH: DỊCH PIVOT VÀ ĐÁNH GIÁ (BLEU, chrF++, COMET)
# ==========================================
def run_pivot_pipeline(dataset, model_name="facebook/nllb-200-3.3B", batch_size=64, output_file="pivot_results.json"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_data = dataset['test']
    
    # Danh sách các cặp ngôn ngữ cần test (sẽ chạy 2 chiều)
    base_pairs = [
        ('en', 'khm'), ('en', 'th'), ('en', 'id'),
        ('fil', 'ms'), ('fil', 'th'), ('fil', 'lo'),
        ('th', 'my'),
        ('lo', 'my')
    ]
    
    # Tạo danh sách 2 chiều
    directions = []
    for src, tgt in base_pairs:
        directions.append((src, tgt))
        directions.append((tgt, src))

    # 1. KHỞI TẠO MODEL DỊCH
    print("--- ĐANG KHỞI TẠO MÔ HÌNH DỊCH THUẬT ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    
    PERT = True
    if PERT:
        print("Đang tải PEFT adapter...")
        try:
            trans_model = PeftModel.from_pretrained(trans_model, "NLLB_3.3B_ALT_SimPO_CPO_v2/models_output/checkpoint-7000")
            trans_model.merge_and_unload()
        except Exception as e:
            print(f"Lỗi khi load PEFT: {e}. Sẽ sử dụng base model.")
            
    trans_model.eval()

    # Khởi tạo sacrebleu metrics
    bleu_metric = BLEU(effective_order=True)
    # chrF++ yêu cầu tham số word_order=2 (mặc định word_order=0 là chrF thường)
    chrf_pp_metric = CHRF(word_order=2) 
    
    # Khởi tạo COMET
    print("--- ĐANG KHỞI TẠO MÔ HÌNH COMET ---")
    comet_model_path = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(comet_model_path)
    gpus_for_comet = 1 if torch.cuda.is_available() else 0

    # 2. CẤU TRÚC LƯU TRỮ KẾT QUẢ
    results = {}
    all_translations_with_scores = []
    
    # 3. CHẠY VÒNG LẶP CHO TỪNG CẶP NGÔN NGỮ
    for src_lang, tgt_lang in directions:
        pair_name = f"{src_lang}_to_{tgt_lang}"
        print(f"\n==============================================")
        print(f"BẮT ĐẦU XỬ LÝ: {src_lang.upper()} -> {tgt_lang.upper()}")
        print(f"==============================================")
        
        src_texts = test_data[src_lang]
        ref_texts = test_data[tgt_lang]
        
        pred_direct = []
        pred_pivot_vi = [] 
        pred_pivot_final = [] 
        
        # --- BƯỚC 1: Dịch trực tiếp (Direct) ---
        for i in tqdm(range(0, len(src_texts), batch_size), desc=f"Direct ({src_lang}->{tgt_lang})"):
            batch_src = src_texts[i:i+batch_size]
            batch_pred = translate_batch(batch_src, tokenizer, trans_model, src_lang, tgt_lang, device)
            pred_direct.extend(batch_pred)
            
        torch.cuda.empty_cache()
            
        # --- BƯỚC 2: Dịch Pivot (src -> vi -> tgt) ---
        # 2a. src -> vi
        for i in tqdm(range(0, len(src_texts), batch_size), desc=f"Pivot 1/2 ({src_lang}->vi)"):
            batch_src = src_texts[i:i+batch_size]
            batch_vi = translate_batch(batch_src, tokenizer, trans_model, src_lang, 'vi', device)
            pred_pivot_vi.extend(batch_vi)
            
        torch.cuda.empty_cache()
            
        # 2b. vi -> tgt
        for i in tqdm(range(0, len(pred_pivot_vi), batch_size), desc=f"Pivot 2/2 (vi->{tgt_lang})"):
            batch_vi = pred_pivot_vi[i:i+batch_size]
            batch_tgt = translate_batch(batch_vi, tokenizer, trans_model, 'vi', tgt_lang, device)
            pred_pivot_final.extend(batch_tgt)
            
        torch.cuda.empty_cache()

        # --- BƯỚC 3: Đánh giá BLEU & chrF++ ---
        score_direct_bleu = bleu_metric.corpus_score(pred_direct, [ref_texts]).score
        score_pivot_bleu = bleu_metric.corpus_score(pred_pivot_final, [ref_texts]).score

        score_direct_chrf = chrf_pp_metric.corpus_score(pred_direct, [ref_texts]).score
        score_pivot_chrf = chrf_pp_metric.corpus_score(pred_pivot_final, [ref_texts]).score
        
        # --- BƯỚC 4: Đánh giá COMET ---
        print("Đang tính điểm COMET...")
        comet_data_direct = [
            {"src": src, "mt": mt, "ref": ref} 
            for src, mt, ref in zip(src_texts, pred_direct, ref_texts)
        ]
        comet_data_pivot = [
            {"src": src, "mt": mt, "ref": ref} 
            for src, mt, ref in zip(src_texts, pred_pivot_final, ref_texts)
        ]
        
        comet_out_direct = comet_model.predict(comet_data_direct, batch_size=batch_size, gpus=gpus_for_comet)
        comet_out_pivot = comet_model.predict(comet_data_pivot, batch_size=batch_size, gpus=gpus_for_comet)
        
        score_direct_comet = comet_out_direct.system_score
        score_pivot_comet = comet_out_pivot.system_score
        
        # Lưu vào dict
        results[pair_name] = {
            'Direct_BLEU': score_direct_bleu,
            'Pivot_BLEU': score_pivot_bleu,
            'Diff_BLEU': score_pivot_bleu - score_direct_bleu,

            'Direct_chrF++': score_direct_chrf,
            'Pivot_chrF++': score_pivot_chrf,
            'Diff_chrF++': score_pivot_chrf - score_direct_chrf,

            'Direct_COMET': score_direct_comet,
            'Pivot_COMET': score_pivot_comet,
            'Diff_COMET': score_pivot_comet - score_direct_comet
        }
        
        # In kết quả
        print(f"\n> KẾT QUẢ {src_lang.upper()} -> {tgt_lang.upper()}:")
        print(f"  [Direct] BLEU: {score_direct_bleu:.2f} | chrF++: {score_direct_chrf:.2f} | COMET: {score_direct_comet:.4f}")
        print(f"  [Pivot]  BLEU: {score_pivot_bleu:.2f} | chrF++: {score_pivot_chrf:.2f} | COMET: {score_pivot_comet:.4f}")
        print(f"  [Diff]   BLEU: {score_pivot_bleu - score_direct_bleu:+.2f} | chrF++: {score_pivot_chrf - score_direct_chrf:+.2f} | COMET: {score_pivot_comet - score_direct_comet:+.4f}")
        
        # Lưu kết quả chi tiết từng câu
        for idx in range(len(src_texts)):
            # Tính chrF++ và BLEU cho từng câu (sentence-level)
            sent_direct_bleu = bleu_metric.sentence_score(pred_direct[idx], [ref_texts[idx]]).score
            sent_pivot_bleu = bleu_metric.sentence_score(pred_pivot_final[idx], [ref_texts[idx]]).score
            sent_direct_chrf = chrf_pp_metric.sentence_score(pred_direct[idx], [ref_texts[idx]]).score
            sent_pivot_chrf = chrf_pp_metric.sentence_score(pred_pivot_final[idx], [ref_texts[idx]]).score

            all_translations_with_scores.append({
                'pair': pair_name,
                'src_text': src_texts[idx],
                'ref_text': ref_texts[idx],
                'pred_direct': pred_direct[idx],
                'pred_pivot_vi': pred_pivot_vi[idx],
                'pred_pivot_final': pred_pivot_final[idx],
                'scores_direct': {
                    'bleu': sent_direct_bleu,
                    'chrf_pp': sent_direct_chrf,
                    'comet': comet_out_direct.scores[idx]
                },
                'scores_pivot': {
                    'bleu': sent_pivot_bleu,
                    'chrf_pp': sent_pivot_chrf,
                    'comet': comet_out_pivot.scores[idx]
                }
            })

        # Backup kết quả liên tục
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump({
                'summary_scores': results,
                'translations': all_translations_with_scores
            }, f, ensure_ascii=False, indent=2)

    print("\n==============================================")
    print(f"HOÀN TẤT! Đã lưu kết quả vào: {output_file}")

# ==========================================
# THỰC THI CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    DATASET_FILE = "DATA/ALT/alt_multilingual_dataset/test.jsonl"
    FINAL_OUTPUT_FILE = "pivot_full_metrics_results.json"
    
    print("Đang tải dữ liệu từ file JSONL...")
    dataset = load_dataset("json", data_files={"test": DATASET_FILE})
    
    run_pivot_pipeline(
        dataset=dataset,
        model_name="facebook/nllb-200-3.3B",
        batch_size=128,  
        output_file=FINAL_OUTPUT_FILE
    )